In [1]:
# ============================================================
# PART 0 — Setup (Kaggle)
# ============================================================
import warnings; warnings.filterwarnings("ignore")
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # before `import torch`

import sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

sns.set_style("whitegrid")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

!pip install pennylane pennylane-lightning --upgrade -q

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA available: {torch.cuda.is_available()}  device: {DEVICE}")

# --- EDIT THESE PATHS FOR YOUR KAGGLE SESSION ---------------------
DATASET = "CICIoT2023"
SCRIPTS_PARENT_DIR = "/kaggle/input/datasets/lawunnannda/quantum-sentinel-scripts"         
DATA_DIR           = f"/kaggle/input/datasets/lawunnannda/quantum-sentinel-iot-v1-0/FROZEN/{DATASET}"
CHECKPOINT_PATH    = "/kaggle/input/models/lawunnannda/qsentinel-models/pytorch/default/8/final-ciciot2023-maqt-train-checkpoint.pt"
HANDOFF_PATH = Path("/kaggle/input/datasets/lawunnannda/quantum-sentinel-iot-v1-0/team-artifacts/teamC_week3_FROZEN_handoff.json")
# --------------------------------------------------------------------------

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 9.2 MB/s eta 0:00:00
CUDA available: False  device: cpu


In [2]:
# ============================================================
# Load MAQT Part-1 Output (Handoff + Cache)
# ============================================================
# after attaching part-1 Notebook Output as a Kaggle dataset:
for p in Path("/kaggle/input").rglob("part1_handoff.pt"):
    print(p)
for p in Path("/kaggle/input").rglob("forward_cache_latest.pt"):
    print(p)

PART1_OUT = Path("/kaggle/input/notebooks/lawunnannda/the-final-ciciot2023-maqt-results-part1/quantum-sentinel")
CACHE_FROM_PART1 = PART1_OUT / "caches" / "forward_cache_latest.pt"
PART1_HANDOFF_PATH = PART1_OUT / "checkpoints" / "part1_handoff.pt"
PART1_TABLES = PART1_OUT / "tables"

assert CACHE_FROM_PART1.exists(), CACHE_FROM_PART1
assert PART1_HANDOFF_PATH.exists(), PART1_HANDOFF_PATH

/kaggle/input/notebooks/lawunnannda/the-final-ciciot2023-maqt-results-part1/quantum-sentinel/checkpoints/part1_handoff.pt
/kaggle/input/notebooks/lawunnannda/the-final-ciciot2023-maqt-results-part1/quantum-sentinel/caches/forward_cache_latest.pt


In [3]:
if SCRIPTS_PARENT_DIR not in sys.path:
    sys.path.append(SCRIPTS_PARENT_DIR)
import scripts
print("scripts loaded from:", scripts.__file__)

from scripts.constants import (
    DEFAULT_ALPHA, DEFAULT_CF, DEFAULT_NOISE_RATE, DEFAULT_REUPLOAD, ZERO_DAY,
    DEFAULT_BETA, PROP1_RESIDUAL_TOL, INPUT_DIM_D, DEFAULT_LOWER_PERCENTILE,
    DEFAULT_UPPER_PERCENTILE,
)
from scripts.data import (
    load_split, capped_sample, greedy_dpp_sample, to_angles,
    class_balance_table, plot_class_balance_bars, plot_class_balance_pie,
)
from scripts.circuit import create_quantum_device, build_forward_circuit
from scripts.prototypes import PrototypeBank
from scripts.quantum_metrics import fidelity, fidelity_pairwise, trace_distance, stack_prototypes
from scripts.inference import qsnet_infer_batch
from scripts.conformal import (
    min_calibration_size, class_conditional_calibrate, per_class_empirical_far,
    threshold_from_scores,
)
from scripts.utils import to_np_batch_x, to_np_y, to_torch_batch_x, expectations_to_tensor
from scripts.logging import to_jsonable, append_jsonl
from scripts.theory import (
    verify_fidelity_convention, check_proposition1_real_data, assert_proposition1,
    analytic_lipschitz_bound, check_lipschitz_tightness,
    linf_to_l2_budget, l2_to_linf_budget, is_eps_safe_to_plot, clopper_pearson_ci,
    worst_case_f_in, quantile_f_out, proposition2_epsilon_star, proposition2_epsilon_beta,
    robust_f_in, robust_f_out, proposition2_epsilon_robust,
    fgsm_gradient_sign, apply_fgsm_perturbation, two_sample_discriminability_auroc,
)
from scripts.cache import (
    ForwardCache, cached_f_max, cached_nonconformity_scores, cached_calibrate_threshold,
    cached_conformal_alpha_sweep, cached_predict_labels, cached_qsnet_infer,
    cached_lipschitz_percentile, cached_qsnet_infer_per_class,
)
from scripts.hilbert import (
    hilbert_geometry_diagnostics, print_h1_report, fidelity_to_prototypes_matrix, pca_2d,
)
from scripts.memory import (
    is_oom_error, is_fatal_cuda_error, safe_empty_cache, run_batched_safely, gpu_memory_snapshot,
)
from scripts.attacks import fgsm_attack, pgd_attack, eval_attacked, robustness_ablation
from scripts.train import train_plain_vqc  # only used in the optional Day-26 ablation block

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# --- Kaggle output layout ---------------------------------------------------
OUT_ROOT   = Path("/kaggle/working/quantum-sentinel")
CACHE_DIR  = OUT_ROOT / "caches"
CKPT_DIR   = OUT_ROOT / "checkpoints"
LOG_DIR    = OUT_ROOT / "logs"
FIG_DIR    = OUT_ROOT / "figures"
TABLE_DIR  = OUT_ROOT / "tables"
for d in (CACHE_DIR, CKPT_DIR, LOG_DIR, FIG_DIR, TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

def savefig(name):
    p = FIG_DIR / f"{name}.png"
    plt.savefig(p, dpi=150, bbox_inches="tight")
    print(f"  saved figure -> {p}")
    plt.close()

def savejson(obj, name, subdir=LOG_DIR):
    p = Path(subdir) / f"{name}.json"
    with open(p, "w") as f:
        json.dump(to_jsonable(obj), f, indent=2)
    print(f"  saved json -> {p}")
    return p

def savecsv(df, name, subdir=TABLE_DIR):
    p = Path(subdir) / f"{name}.csv"
    df.to_csv(p, index=False)
    print(f"  saved table -> {p}")
    return p

check = verify_fidelity_convention(dim=4, n_trials=30, seed=SEED)
print("Fidelity-convention self-check PASSED (synthetic unit test only):", check)

scripts loaded from: /kaggle/input/datasets/lawunnannda/quantum-sentinel-scripts/scripts/__init__.py
Fidelity-convention self-check PASSED (synthetic unit test only): {'max_pairwise_disagreement': 1.5543122344752192e-15, 'max_fvg_violation': 0.0}


In [4]:
# ============================================================
# PART 1 — Load checkpoint (theta_star, head, prototypes, config)
# ============================================================
print("="*60); print("LOADING CHECKPOINT"); print("="*60)
assert Path(CHECKPOINT_PATH).exists(), f"Not found: {CHECKPOINT_PATH}"
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)

theta_star      = ckpt["theta"]
head_state_dict = ckpt["head_state_dict"]
prototypes_raw  = ckpt["prototypes"]
class_names     = ckpt["class_names"]
num_classes     = ckpt["num_classes"]
num_qubits      = ckpt["num_qubits"]
num_layers      = ckpt["num_layers"]
noise_rate      = ckpt.get("noise_rate", DEFAULT_NOISE_RATE)
reupload        = ckpt.get("reupload", DEFAULT_REUPLOAD)
feature_cols    = ckpt["feature_cols"]
use_pca         = ckpt.get("use_pca", True)
target_col      = "label_multiclass"

scaler   = ckpt["scaler"]
pca      = ckpt.get("pca", None)
angle_x_min = np.asarray(ckpt["angle_x_min"])
angle_x_max = np.asarray(ckpt["angle_x_max"])
angle_max   = ckpt.get("angle_max", float(np.pi))

SUBSET        = ckpt.get("subset", True)
PER_CLASS_CAP = ckpt.get("per_class_cap", 7000)
SAMPLER       = ckpt.get("sampler", "greedy_dpp")

prototypes = {int(k): (v if torch.is_tensor(v) else torch.tensor(v)).to(DEVICE)
              for k, v in prototypes_raw.items()}

print(f" classes    : {class_names}")
print(f" qubits     : {num_qubits}  layers: {num_layers}  reupload: {reupload}  noise: {noise_rate}")
print(f" theta      : {tuple(theta_star.shape)}")
print(f" prototypes : {len(prototypes)} classes")

classifier_head = nn.Linear(num_qubits, num_classes).to(DEVICE)
classifier_head.load_state_dict(head_state_dict)
classifier_head.eval()
print(f" head       : Linear({num_qubits} -> {num_classes}) loaded OK")

LOADING CHECKPOINT
 classes    : ['Backdoor_Malware', 'BenignTraffic', 'BrowserHijacking', 'CommandInjection', 'DDoS-ACK_Fragmentation', 'DDoS-HTTP_Flood', 'DDoS-ICMP_Flood', 'DDoS-ICMP_Fragmentation', 'DDoS-PSHACK_Flood', 'DDoS-RSTFINFlood', 'DDoS-SYN_Flood', 'DDoS-SlowLoris', 'DDoS-SynonymousIP_Flood', 'DDoS-TCP_Flood', 'DDoS-UDP_Flood', 'DDoS-UDP_Fragmentation', 'DNS_Spoofing', 'DictionaryBruteForce', 'DoS-HTTP_Flood', 'DoS-SYN_Flood', 'DoS-TCP_Flood', 'DoS-UDP_Flood', 'MITM-ArpSpoofing', 'Recon-HostDiscovery', 'Recon-OSScan', 'Recon-PingSweep', 'Recon-PortScan', 'SqlInjection', 'Uploading_Attack', 'VulnerabilityScan', 'XSS']
 qubits     : 6  layers: 3  reupload: True  noise: 0.01
 theta      : (3, 6, 3)
 prototypes : 31 classes
 head       : Linear(6 -> 31) loaded OK


In [5]:
# ============================================================
# Verify Team C's FROZEN features
# ============================================================
print("="*60); print("VERIFYING FROZEN FEATURES"); print("="*60)

handoff = json.loads(HANDOFF_PATH.read_text())
teamc_cols = list(handoff["frozen_subsets"][DATASET]["features"])
ckpt_cols = list(ckpt["feature_cols"])
print(f"Team C selector: {handoff['frozen_subsets'][DATASET]['selector']}\n")
print(f"Team C k={len(teamc_cols)}: {teamc_cols}\n")
print(f"ckpt   k={len(ckpt_cols)}: {ckpt_cols}\n")

same_set = set(teamc_cols) == set(ckpt_cols)
same_order = teamc_cols == ckpt_cols
print(f"same features (set):   {same_set}")
print(f"same order (list):     {same_order}")
print()
if not same_set:
    only_teamc = sorted(set(teamc_cols) - set(ckpt_cols))
    only_ckpt = sorted(set(ckpt_cols) - set(teamc_cols))
    print(f"  only in Team C: {only_teamc}")
    print(f"  only in ckpt:   {only_ckpt}")
    raise ValueError("checkpoint feature_cols do not match Team C FROZEN subset")
if not same_order:
    print("warning: same columns, different order — encoding may still break if order mattered at train time")

VERIFYING FROZEN FEATURES
Team C selector: MI

Team C k=10: ['duration', 'n_pkts_total', 'n_bytes_total', 'rate', 'pkt_size_min', 'pkt_size_max', 'pkt_size_mean', 'iat', 'protocol', 'conn_state']

ckpt   k=10: ['duration', 'n_pkts_total', 'n_bytes_total', 'rate', 'pkt_size_min', 'pkt_size_max', 'pkt_size_mean', 'iat', 'protocol', 'conn_state']

same features (set):   True
same order (list):     True



In [6]:
# ============================================================
# PART 2 — Load raw splits, apply the SAME subsampling/encoding as training
# ============================================================
print("="*60); print("LOADING RAW DATA SPLITS"); print("="*60)

X_train_full, y_train_full, _ = load_split(DATA_DIR, "train", target_col, csv=True, selected_cols=feature_cols)
X_test,       y_test,       _ = load_split(DATA_DIR, "test", target_col, csv=True, selected_cols=feature_cols)
X_cal,        y_cal,        _ = load_split(DATA_DIR, "calibration", target_col, csv=True, selected_cols=feature_cols)
X_zeroday,    y_zeroday,    _ = load_split(DATA_DIR, "zeroday", target_col, csv=True, selected_cols=feature_cols)

print(f" train(full): {X_train_full.shape}  test: {X_test.shape}  cal: {X_cal.shape}  zeroday: {X_zeroday.shape}")

if SUBSET and SAMPLER == "greedy_dpp":
    X_train, y_train = greedy_dpp_sample(X_train_full, y_train_full, per_class_cap=PER_CLASS_CAP, seed=SEED)
elif SUBSET:
    X_train, y_train = capped_sample(X_train_full, y_train_full, per_class_cap=PER_CLASS_CAP, seed=SEED)
else:
    X_train, y_train = X_train_full, y_train_full
print(f" train(subset): {X_train.shape} via {SAMPLER}")

def encode(X):
    return to_angles(X, scaler, angle_x_min, angle_x_max,
                      pca=pca if use_pca else None, angle_max=angle_max)

A_train, A_test, A_cal, A_zeroday = encode(X_train), encode(X_test), encode(X_cal), encode(X_zeroday)
print(f" A_train {A_train.shape}  A_test {A_test.shape}  A_cal {A_cal.shape}  A_zeroday {A_zeroday.shape}")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
plot_class_balance_bars(y_train, class_names, title="Train (subset) class balance", ax=ax[0])
plot_class_balance_pie(y_train, class_names, title="Train (subset) class balance", ax=ax[1])
plt.tight_layout(); savefig("part2_class_balance")

LOADING RAW DATA SPLITS
 train(full): (151049, 10)  test: (18883, 10)  cal: (18883, 10)  zeroday: (10984, 10)
 train(subset): (83617, 10) via greedy_dpp
 A_train (83617, 6)  A_test (18883, 6)  A_cal (18883, 6)  A_zeroday (10984, 6)
  saved figure -> /kaggle/working/quantum-sentinel/figures/part2_class_balance.png


In [7]:
# ============================================================
# PART 3 — Quantum device + ForwardCache (ONE forward pass per split, ever)
# ============================================================
print("="*60); print("QUANTUM CIRCUIT"); print("="*60)

dev = create_quantum_device(num_qubits)
forward_circuit = build_forward_circuit(dev, num_qubits, num_layers,
                                         noise_rate=noise_rate, reupload=reupload)

theta_device = theta_star.to(DEVICE)
print(f" device={DEVICE}  wires={num_qubits}  theta on {theta_device.device}")

CACHE_LATEST = CACHE_FROM_PART1
assert CACHE_LATEST.exists(), CACHE_LATEST

cache = ForwardCache(store_device="cpu")
print(f"LOADING forward cache from part 1: {CACHE_LATEST}")
cache_data = torch.load(CACHE_LATEST, map_location="cpu", weights_only=False)
for key in ["train", "test", "cal", "zeroday"]:
    cache._entries[key] = {
        "z": cache_data[f"{key}_z"], "rho": cache_data[f"{key}_rho"],
        "n": cache_data[f"{key}_n"], "p": noise_rate, "X": cache_data[f"{key}_X"],
    }

for row in cache.memory_report():
    print(f"  {row['key']:8s}: n={row['n_samples']:5d}  {row['MB']:7.1f} MB (CPU)")

QUANTUM CIRCUIT
 device=cpu  wires=6  theta on cpu
LOADING forward cache from part 1: /kaggle/input/notebooks/lawunnannda/the-final-ciciot2023-maqt-results-part1/quantum-sentinel/caches/forward_cache_latest.pt
  train   : n=83617   5481.9 MB (CPU)
  test    : n=18883   1238.0 MB (CPU)
  cal     : n=18883   1238.0 MB (CPU)
  zeroday : n=10984    720.1 MB (CPU)


In [8]:
# ============================================================
# HYDRATE — load part1_handoff.pt (replaces PART 4–5 + Days 15–18b)
# ============================================================
HANDOFF = torch.load(PART1_HANDOFF_PATH, map_location="cpu", weights_only=False)

# safety: part 2 data load must match part 1
assert HANDOFF["dataset"] == DATASET
assert HANDOFF["seed"] == SEED
assert np.array_equal(y_test, HANDOFF["y_test"]), "y_test mismatch vs part 1 — check PART 2 sampling"

# PART 5
q_final    = HANDOFF["q_final"]
q_by_class = {int(k): float(v) for k, v in HANDOFF["q_by_class"].items()}
calib_meta = HANDOFF.get("calib_meta")

# Days 15–16
L_PHI_ANALYTIC = HANDOFF["L_PHI_ANALYTIC"]
max_resid      = HANDOFF["max_resid"]

# Day 17
labels_test_final       = HANDOFF["labels_test_final"]
radii_test_final        = HANDOFF["radii_test_final"]
scores_test_final       = HANDOFF["scores_test_final"]
certified_radii_correct = HANDOFF["certified_radii_correct"]

# Days 18–18b
F_max_test       = HANDOFF["F_max_test"]
F_max_zeroday    = HANDOFF["F_max_zeroday"]
F_IN             = HANDOFF["F_IN"]
F_OUT            = HANDOFF["F_OUT"]
delta_strict     = HANDOFF["delta_strict"]
eps_star_strict  = HANDOFF["eps_star_strict"]
F_out_beta       = HANDOFF["F_out_beta"]
delta_beta       = HANDOFF["delta_beta"]
eps_beta         = HANDOFF["eps_beta"]
F_in_robust      = HANDOFF["F_in_robust"]
F_out_robust     = HANDOFF["F_out_robust"]
delta_robust     = HANDOFF["delta_robust"]
eps_robust       = HANDOFF["eps_robust"]

# needed by Save Days 15–25 bundle (not in .pt — load CSV)
df_prop2_perclass = pd.read_csv(PART1_TABLES / "day18c_mondrian_proposition2.csv")

print(f"hydrated from part 1: q={q_final:.4f}  L_phi={L_PHI_ANALYTIC:.4f}  "
      f"eps*={eps_star_strict:.4f}  n_test={len(labels_test_final)}")

hydrated from part 1: q=0.2425  L_phi=1.5000  eps*=-0.2848  n_test=18883


In [9]:
# ============================================================
# HYBRID REJECTION — quantum s(x) fused with classical Isolation Forest
# ============================================================
USE_ISO_HYBRID = True
X_train_classical = X_train  # same feature view fed to the encoder (pre angle-encoding)
X_test_classical, X_cal_classical, X_zeroday_classical = X_test, X_cal, X_zeroday

if USE_ISO_HYBRID:
    iso_clf = IsolationForest(n_estimators=150, contamination="auto", random_state=SEED)
    iso_clf.fit(X_train_classical)
    def iso_nonconformity(X): return -iso_clf.score_samples(X)

    iso_cal, iso_test, iso_zday = (iso_nonconformity(X_cal_classical),
                                    iso_nonconformity(X_test_classical),
                                    iso_nonconformity(X_zeroday_classical))

    q_score_cal  = cached_nonconformity_scores(cache.get("cal"), prototypes, device=DEVICE, batch_size=256)
    q_score_test = cached_nonconformity_scores(cache.get("test"), prototypes, device=DEVICE, batch_size=256)
    q_score_zday = cached_nonconformity_scores(cache.get("zeroday"), prototypes, device=DEVICE, batch_size=256)

    def percentile_rank(cal_ref, values):
        cal_sorted = np.sort(cal_ref)
        return np.searchsorted(cal_sorted, values, side="right") / len(cal_sorted)

    combined_cal  = np.maximum(percentile_rank(q_score_cal, q_score_cal),  percentile_rank(iso_cal, iso_cal))
    combined_test = np.maximum(percentile_rank(q_score_cal, q_score_test), percentile_rank(iso_cal, iso_test))
    combined_zday = np.maximum(percentile_rank(q_score_cal, q_score_zday), percentile_rank(iso_cal, iso_zday))

    q_hybrid, _ = threshold_from_scores(combined_cal, alpha=DEFAULT_ALPHA)
    tp_h, n_h = int(np.sum(combined_zday > q_hybrid)), len(combined_zday)
    fp_h, m_h = int(np.sum(combined_test > q_hybrid)), len(combined_test)
    TPR_hybrid, FPR_hybrid = tp_h/n_h, fp_h/m_h
    tpr_h_lo, tpr_h_hi = clopper_pearson_ci(tp_h, n_h, alpha=0.05)
    fpr_h_lo, fpr_h_hi = clopper_pearson_ci(fp_h, m_h, alpha=0.05)
    print(f"[hybrid] TPR={TPR_hybrid:.4f} CI[{tpr_h_lo:.4f},{tpr_h_hi:.4f}]  "
          f"FPR={FPR_hybrid:.4f} CI[{fpr_h_lo:.4f},{fpr_h_hi:.4f}]")

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].hist(q_score_test, bins=30, alpha=0.6, label="quantum, test")
    ax[0].hist(q_score_zday, bins=30, alpha=0.6, label="quantum, zero-day"); ax[0].legend(fontsize=8)
    ax[1].hist(iso_test, bins=30, alpha=0.6, label="IsoForest, test")
    ax[1].hist(iso_zday, bins=30, alpha=0.6, label="IsoForest, zero-day"); ax[1].legend(fontsize=8)
    plt.tight_layout(); savefig("hybrid_scores")

[hybrid] TPR=0.3719 CI[0.3629,0.3810]  FPR=0.0471 CI[0.0441,0.0502]
  saved figure -> /kaggle/working/quantum-sentinel/figures/hybrid_scores.png


In [10]:
# ============================================================
# DAY 19 — FGSM epsilon sweep (gradient computed ONCE, reused for whole sweep)
# ============================================================
accepted_idx = np.where(labels_test_final != ZERO_DAY)[0][:300]
A_known_accepted = A_test[accepted_idx]
_, proto_stack_main = stack_prototypes(prototypes)

grad_sign_test = run_batched_safely(fgsm_gradient_sign, A_known_accepted, theta_device, prototypes,
                                     forward_circuit, device=DEVICE, proto_stack=proto_stack_main,
                                     batch_size=8, min_batch=1, label="fgsm_grad[test]")

eps_inf_list = [0.0, 0.01, 0.05, 0.08, 0.1, 0.2, 0.3]
day19_rows = []
for eps_inf in eps_inf_list:
    eps_l2 = linf_to_l2_budget(eps_inf, d=num_qubits)
    if eps_inf == 0.0:
        infer_out = cached_qsnet_infer(cache.slice("test", accepted_idx), prototypes, q_final,
                                        p=noise_rate, L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF,
                                        zero_day=ZERO_DAY, device=DEVICE, batch_size=256)
    elif grad_sign_test is None:
        infer_out = None
    else:
        X_adv = apply_fgsm_perturbation(A_known_accepted, grad_sign_test, eps_inf, device=DEVICE,
                                         x_min=0.0, x_max=float(np.pi))
        infer_out = run_batched_safely(qsnet_infer_batch, X_adv.cpu().numpy(), theta_device, prototypes,
                                        q_final, forward_circuit, p=noise_rate, L_phi=L_PHI_ANALYTIC,
                                        Cf=DEFAULT_CF, zero_day=ZERO_DAY, device=DEVICE,
                                        batch_size=8, min_batch=1, label=f"infer[eps={eps_inf}]")
    if infer_out is None:
        day19_rows.append({"eps_inf": eps_inf, "eps_l2": eps_l2, "acceptance_rate": np.nan}); continue
    day19_rows.append({"eps_inf": eps_inf, "eps_l2": eps_l2,
                        "acceptance_rate": float(np.mean(infer_out[0] != ZERO_DAY))})
    safe_empty_cache()

df_day19 = pd.DataFrame(day19_rows).dropna(subset=["acceptance_rate"])
below = df_day19[df_day19["acceptance_rate"] < 0.99]
empirical_breakpoint_l2 = float(below["eps_l2"].iloc[0]) if len(below) else float("inf")
print(f"Empirical breakpoint (L2, acceptance<99%): {empirical_breakpoint_l2:.4f}")
savecsv(df_day19, "day19_fgsm_sweep")

eps_plot, is_safe = is_eps_safe_to_plot(eps_star_strict)
plt.figure(figsize=(8, 5))
plt.plot(df_day19["eps_l2"], df_day19["acceptance_rate"], marker="o", label="empirical acceptance")
if is_safe:
    plt.axvline(eps_plot, color="green", linestyle="--", label=f"certified eps*={eps_plot:.3f}")
else:
    plt.text(0.02, 0.02, f"eps*={eps_star_strict:.3f} (<=0, not separable)",
              transform=plt.gca().transAxes, color="green")
if np.isfinite(empirical_breakpoint_l2):
    plt.axvline(empirical_breakpoint_l2, color="red", linestyle=":", label="empirical breakpoint")
plt.legend(); plt.title("Day 19: acceptance vs attack budget")
plt.tight_layout(); savefig("day19_fgsm_sweep")

Empirical breakpoint (L2, acceptance<99%): 0.1960
  saved table -> /kaggle/working/quantum-sentinel/tables/day19_fgsm_sweep.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/day19_fgsm_sweep.png


In [11]:
# ============================================================
# DAY 20 — H3 (empirical breakpoint >= certified eps*) + noise-rate p sweep
# ============================================================
h3_holds = np.isfinite(empirical_breakpoint_l2) and empirical_breakpoint_l2 >= eps_star_strict
print(f"H3 holds: {h3_holds}" + ("  [VACUOUS: eps*<=0]" if eps_star_strict <= 0 else ""))

p_sweep_values = [0.0,0.05, 0.1, 0.2]
eps_inf_small = [0.0, 0.05, 0.1, 0.2]
day20_rows = []
for p_val in p_sweep_values:
    print(f"p={p_val:.2f}")
    fc_p = theta_dev = prototypes_p = p_cache = None
    try:
        fc_p = build_forward_circuit(dev, num_qubits, num_layers, noise_rate=p_val, reupload=reupload)
        theta_dev = theta_device
        proto_bank_p = PrototypeBank(classes=range(num_classes))
        prototypes_p = run_batched_safely(proto_bank_p.compute, theta_dev, A_train, y_train,
                                           forward_circuit=fc_p, device=DEVICE, batch_size=8,
                                           min_batch=1, label=f"protos[p={p_val}]")
        if prototypes_p is None:
            raise RuntimeError("OOM (prototype computation skipped)")

        p_cache = ForwardCache(store_device="cpu")
        test_entry = run_batched_safely(p_cache.compute, "test", A_test, theta_dev, fc_p, device=DEVICE,
                                         p=p_val, batch_size=8, min_batch=1, label=f"cache_test[p={p_val}]")
        zday_entry = run_batched_safely(p_cache.compute, "zeroday", A_zeroday, theta_dev, fc_p, device=DEVICE,
                                         p=p_val, batch_size=8, min_batch=1, label=f"cache_zday[p={p_val}]")
        if test_entry is None or zday_entry is None:
            raise RuntimeError("OOM (per-p cache build skipped)")

        F_max_test_p = cached_f_max(test_entry, prototypes_p, device=DEVICE, batch_size=256)
        F_max_zday_p = cached_f_max(zday_entry, prototypes_p, device=DEVICE, batch_size=256)
        F_in_p, F_out_p = worst_case_f_in(F_max_test_p, percentile=0.0), float(F_max_zday_p.max())
        _, eps_star_p       = proposition2_epsilon_star(F_in_p, F_out_p, p=p_val, L_phi=L_PHI_ANALYTIC)
        _, eps_star_naive_p = proposition2_epsilon_star(F_IN, F_OUT, p=p_val, L_phi=L_PHI_ANALYTIC)

        labels_base_p, *_ = cached_qsnet_infer(test_entry, prototypes_p, q_final, p=p_val,
                                                L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF, zero_day=ZERO_DAY,
                                                device=DEVICE, batch_size=256)
        accepted_p_idx = np.where(labels_base_p != ZERO_DAY)[0][:200]
        accepted_p = A_test[accepted_p_idx]
        _, proto_stack_p = stack_prototypes(prototypes_p)
        grad_sign_p = run_batched_safely(fgsm_gradient_sign, accepted_p, theta_dev, prototypes_p, fc_p,
                                          device=DEVICE, proto_stack=proto_stack_p, batch_size=8,
                                          min_batch=1, label=f"fgsm_grad[p={p_val}]") if len(accepted_p) else None

        breakpoint_p = float("inf")
        for eps_inf in eps_inf_small:
            if eps_inf == 0.0:
                infer_adv = cached_qsnet_infer(p_cache.slice("test", accepted_p_idx), prototypes_p, q_final,
                                                p=p_val, L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF,
                                                zero_day=ZERO_DAY, device=DEVICE, batch_size=256)
            elif grad_sign_p is None:
                continue
            else:
                X_adv_p = apply_fgsm_perturbation(accepted_p, grad_sign_p, eps_inf, device=DEVICE,
                                                   x_min=0.0, x_max=float(np.pi))
                infer_adv = run_batched_safely(qsnet_infer_batch, X_adv_p.cpu().numpy(), theta_dev,
                                                prototypes_p, q_final, fc_p, p=p_val, L_phi=L_PHI_ANALYTIC,
                                                Cf=DEFAULT_CF, zero_day=ZERO_DAY, device=DEVICE,
                                                batch_size=8, min_batch=1, label=f"infer_adv[p={p_val},eps={eps_inf}]")
            if infer_adv is not None and float(np.mean(infer_adv[0] != ZERO_DAY)) < 0.99:
                breakpoint_p = linf_to_l2_budget(eps_inf, d=num_qubits); break

        day20_rows.append({"p": p_val, "F_in": F_in_p, "F_out": F_out_p,
                            "epsilon_star_correct": eps_star_p, "epsilon_star_naive_wrong": eps_star_naive_p,
                            "empirical_breakpoint_l2": breakpoint_p})
        print(f"  F_in={F_in_p:.4f} F_out={F_out_p:.4f} eps*_correct={eps_star_p:.4f} "
              f"eps*_naive={eps_star_naive_p:.4f} breakpoint={breakpoint_p:.4f}")
    except RuntimeError as e:
        if is_fatal_cuda_error(e):
            print("  [FATAL] non-recoverable CUDA error — stopping p-sweep."); break
        elif is_oom_error(e):
            print(f"  [OOM] p={p_val} skipped."); day20_rows.append({"p": p_val, "F_in": np.nan,
                  "F_out": np.nan, "epsilon_star_correct": np.nan,
                  "epsilon_star_naive_wrong": np.nan, "empirical_breakpoint_l2": np.nan})
        else:
            raise
    finally:
        del fc_p, theta_dev, prototypes_p, p_cache
        safe_empty_cache()

df_day20 = pd.DataFrame(day20_rows)
savecsv(df_day20, "day20_p_sweep")

df_valid = df_day20.dropna(subset=["epsilon_star_correct"])
if not df_valid.empty:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].plot(df_valid["p"], df_valid["epsilon_star_correct"], marker="o", label="eps* (re-measured at p)")
    ax[0].plot(df_valid["p"], df_valid["epsilon_star_naive_wrong"], marker="x", linestyle="--",
               color="red", label="eps* (WRONG: fixed at p=0)")
    ax[0].axhline(0, color="gray", linewidth=0.8); ax[0].legend(fontsize=8)
    ax[0].set_title("H4: epsilon* vs p")
    ax[1].plot(df_valid["p"], df_valid["epsilon_star_correct"], marker="o", label="certified eps*")
    ax[1].plot(df_valid["p"], df_valid["empirical_breakpoint_l2"], marker="s", label="empirical breakpoint")
    ax[1].legend(); ax[1].set_title("H3/H4: certified vs empirical")
    plt.tight_layout(); savefig("day20_p_sweep")
    h3_all = (df_valid["empirical_breakpoint_l2"] >= df_valid["epsilon_star_correct"]).all()
    print(f"H3 holds for every p tested: {h3_all}")

H3 holds: True  [VACUOUS: eps*<=0]
p=0.00
  F_in=0.3196 F_out=0.9477 eps*_correct=-0.2984 eps*_naive=-0.2819 breakpoint=0.1225
p=0.05
  F_in=0.7767 F_out=0.9919 eps*_correct=-0.2182 eps*_naive=-0.2967 breakpoint=inf
p=0.10
  F_in=0.9160 F_out=0.9977 eps*_correct=-0.1477 eps*_naive=-0.3132 breakpoint=inf
p=0.20
  F_in=0.9921 F_out=0.9998 eps*_correct=-0.0522 eps*_naive=-0.3524 breakpoint=inf
  saved table -> /kaggle/working/quantum-sentinel/tables/day20_p_sweep.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/day20_p_sweep.png
H3 holds for every p tested: True


In [12]:
# ============================================================
# DAY 21 — Extend Prop-2 quantities to train/test/cal + freeze robustness interface
# ============================================================
f_max_cache_by_name = {"test": F_max_test, "zeroday": F_max_zeroday}
day21_rows = [{"dataset": "test", "role": "known", "F_worst_min": float(F_max_test.min()),
               "F_p1": float(np.percentile(F_max_test, 1)), "F_mean": float(F_max_test.mean())}]
for name in ["train", "cal"]:
    fmax = cached_f_max(cache.get(name), prototypes, device=DEVICE, batch_size=256)
    f_max_cache_by_name[name] = fmax
    day21_rows.append({"dataset": name, "role": "known", "F_worst_min": float(fmax.min()),
                        "F_p1": float(np.percentile(fmax, 1)), "F_mean": float(fmax.mean())})
day21_rows.append({"dataset": "zeroday", "role": "novel", "F_worst_max": float(F_max_zeroday.max()),
                    "F_p95": quantile_f_out(F_max_zeroday, beta=0.05), "F_mean": float(F_max_zeroday.mean())})
df_day21 = pd.DataFrame(day21_rows)
savecsv(df_day21, "day21_prop2_all_datasets")

F_IN_GLOBAL = min(f_max_cache_by_name["train"].min(), f_max_cache_by_name["test"].min(),
                   f_max_cache_by_name["cal"].min())
_, EPS_STAR_GLOBAL = proposition2_epsilon_star(F_IN_GLOBAL, F_OUT, p=noise_rate, L_phi=L_PHI_ANALYTIC)
print(f"Global F_in: {F_IN_GLOBAL:.4f}  Global epsilon*: {EPS_STAR_GLOBAL:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([f_max_cache_by_name[n] for n in ["train", "test", "cal", "zeroday"]],
           labels=["train", "test", "cal", "zeroday"])
ax.axhline(F_IN_GLOBAL, color="green", linestyle="--", label=f"F_in(global)={F_IN_GLOBAL:.3f}")
ax.axhline(F_OUT, color="red", linestyle="--", label=f"F_out(zeroday)={F_OUT:.3f}")
ax.legend(); plt.tight_layout(); savefig("day21_fmax_boxplot")

robustness_interface = {
    "theta": theta_star, "head_state_dict": head_state_dict,
    "prototypes": {c: rho.detach().cpu() for c, rho in prototypes.items()},
    "q_threshold": q_final, "q_by_class": q_by_class, "L_phi_analytic": L_PHI_ANALYTIC,
    "noise_p": noise_rate, "Cf": DEFAULT_CF, "num_qubits": num_qubits, "num_layers": num_layers,
    "F_in_global": F_IN_GLOBAL, "F_out_zeroday_worst": F_OUT, "epsilon_star_global": EPS_STAR_GLOBAL,
    "class_names": class_names, "use_pca": use_pca,
}
torch.save(robustness_interface, CKPT_DIR / "robustness_interface_day21.pt")
print(f"Froze robustness interface -> {CKPT_DIR / 'robustness_interface_day21.pt'}")
safe_empty_cache()

  saved table -> /kaggle/working/quantum-sentinel/tables/day21_prop2_all_datasets.csv
Global F_in: 0.4452  Global epsilon*: -0.2935
  saved figure -> /kaggle/working/quantum-sentinel/figures/day21_fmax_boxplot.png
Froze robustness interface -> /kaggle/working/quantum-sentinel/checkpoints/robustness_interface_day21.pt


In [13]:
# ============================================================
# Hilbert-geometry diagnostics + PCA-of-fidelity-vectors scatter
# ============================================================
h1_report = hilbert_geometry_diagnostics(theta_device, A_train, y_train, prototypes, forward_circuit,
                                          class_names=class_names, device=DEVICE, max_per_class=80,
                                          batch_size=8)
print_h1_report(h1_report)
savejson({k: v for k, v in h1_report.items() if k != "pairs"}, "part16_h1_report")

N_VIZ = 400
rng_viz = np.random.default_rng(SEED)
test_idx_viz = rng_viz.choice(len(A_test), size=min(N_VIZ, len(A_test)), replace=False)
zday_idx_viz = rng_viz.choice(len(A_zeroday), size=min(N_VIZ, len(A_zeroday)), replace=False)
rho_viz = torch.cat([cache.get("test")["rho"][test_idx_viz], cache.get("zeroday")["rho"][zday_idx_viz]], dim=0)
_, feats_viz = fidelity_to_prototypes_matrix(rho_viz, prototypes)
emb_viz = pca_2d(feats_viz)
labels_viz = list(y_test[test_idx_viz]) + [ZERO_DAY] * len(zday_idx_viz)

plt.figure(figsize=(8, 7))
palette = plt.cm.tab10(np.linspace(0, 1, num_classes))
for c in range(num_classes):
    m = np.array([l == c for l in labels_viz])
    plt.scatter(emb_viz[m, 0], emb_viz[m, 1], s=14, alpha=0.7, color=palette[c], label=class_names[c])
m_zday = np.array([l == ZERO_DAY for l in labels_viz])
plt.scatter(emb_viz[m_zday, 0], emb_viz[m_zday, 1], s=20, alpha=0.85, color="black", marker="x", label="zero-day")
plt.legend(fontsize=8); plt.title("PCA of fidelity-to-prototype vectors (test + zero-day)")
plt.tight_layout(); savefig("part16_hilbert_pca_scatter")

=== H1 Hilbert geometry (fidelity gaps) ===
mean intra-class fidelity : 0.7839
mean inter-class fidelity : 0.6233
fidelity gap (intra-inter): 0.1606  ← want ↑
mean inter trace distance : 0.6717  ← want ↑

per-class intra fidelity:
  Backdoor_Malware             n=  75  F=0.6372
  BenignTraffic                n=  80  F=0.6722
  BrowserHijacking             n=  80  F=0.6045
  CommandInjection             n=  80  F=0.6059
  DDoS-ACK_Fragmentation       n=  80  F=0.8923
  DDoS-HTTP_Flood              n=  80  F=0.8490
  DDoS-ICMP_Flood              n=  80  F=0.9833
  DDoS-ICMP_Fragmentation      n=  80  F=0.9559
  DDoS-PSHACK_Flood            n=  80  F=0.9509
  DDoS-RSTFINFlood             n=  80  F=0.9279
  DDoS-SYN_Flood               n=  80  F=0.9452
  DDoS-SlowLoris               n=  80  F=0.8060
  DDoS-SynonymousIP_Flood      n=  80  F=0.9486
  DDoS-TCP_Flood               n=  80  F=0.9406
  DDoS-UDP_Flood               n=  80  F=0.9931
  DDoS-UDP_Fragmentation       n=  80  F=0.9446
 

In [14]:
# ============================================================
# DAY 22 — p(x) -> fidelity -> score -> conformal test -> label + radius
# ============================================================
for name, key, idx in [("known-class test sample", "test", 0), ("zeroday sample", "zeroday", 0)]:
    rho_x = cache.get(key)["rho"][idx].to(DEVICE)
    with torch.no_grad():
        f_map = {class_names[c]: float(fidelity(rho_x, prototypes[c])) for c in sorted(prototypes)}
    f_max = max(f_map.values()); score = 1.0 - f_max; accepted = score <= q_final
    decision = max(f_map, key=f_map.get) if accepted else "ZERO_DAY (rejected)"
    print(f"\n--- {name} ---")
    print(f" fidelity: { {k: round(v,4) for k,v in f_map.items()} }")
    print(f" F_max={f_max:.4f}  s(x)={score:.4f}  q={q_final:.4f}  -> {decision}")


--- known-class test sample ---
 fidelity: {'Backdoor_Malware': 0.3811, 'BenignTraffic': 0.3729, 'BrowserHijacking': 0.408, 'CommandInjection': 0.408, 'DDoS-ACK_Fragmentation': 0.4933, 'DDoS-HTTP_Flood': 0.764, 'DDoS-ICMP_Flood': 0.8384, 'DDoS-ICMP_Fragmentation': 0.4931, 'DDoS-PSHACK_Flood': 0.7434, 'DDoS-RSTFINFlood': 0.7169, 'DDoS-SYN_Flood': 0.8275, 'DDoS-SlowLoris': 0.6136, 'DDoS-SynonymousIP_Flood': 0.8476, 'DDoS-TCP_Flood': 0.9642, 'DDoS-UDP_Flood': 0.6573, 'DDoS-UDP_Fragmentation': 0.4268, 'DNS_Spoofing': 0.4202, 'DictionaryBruteForce': 0.4206, 'DoS-HTTP_Flood': 0.7468, 'DoS-SYN_Flood': 0.8428, 'DoS-TCP_Flood': 0.9543, 'DoS-UDP_Flood': 0.7082, 'MITM-ArpSpoofing': 0.3888, 'Recon-HostDiscovery': 0.4749, 'Recon-OSScan': 0.446, 'Recon-PingSweep': 0.4519, 'Recon-PortScan': 0.463, 'SqlInjection': 0.4089, 'Uploading_Attack': 0.4152, 'VulnerabilityScan': 0.5852, 'XSS': 0.4095}
 F_max=0.9642  s(x)=0.0358  q=0.2425  -> DDoS-TCP_Flood

--- zeroday sample ---
 fidelity: {'Backdoor_Malware

In [15]:
# ============================================================
# DAY 23-24 — RQ1 detection numbers (global threshold, with Clopper-Pearson CIs)
# ============================================================
labels_zday_final, radii_zday_final, scores_zday_final, _ = cached_qsnet_infer(
    cache.get("zeroday"), prototypes, q_final, p=noise_rate, L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF,
    zero_day=ZERO_DAY, device=DEVICE, batch_size=256)

n_zday = len(labels_zday_final); tp_zday = int(np.sum(labels_zday_final == ZERO_DAY))
TPR_zeroday = tp_zday / n_zday
tpr_lo, tpr_hi = clopper_pearson_ci(tp_zday, n_zday, alpha=0.05)

n_known = len(labels_test_final); fp_known = int(np.sum(labels_test_final == ZERO_DAY))
FPR_known = fp_known / n_known
fpr_lo, fpr_hi = clopper_pearson_ci(fp_known, n_known, alpha=0.05)

accepted_known = labels_test_final != ZERO_DAY
class_acc_known = float(np.mean(labels_test_final[accepted_known] == y_test[accepted_known]))
print(f"[global] TPR={TPR_zeroday:.4f} CI[{tpr_lo:.4f},{tpr_hi:.4f}]  "
      f"FPR={FPR_known:.4f} CI[{fpr_lo:.4f},{fpr_hi:.4f}]  class_acc={class_acc_known:.4f}")

# ============================================================
# DAY 23-24b — RQ1 detection numbers under Mondrian per-class thresholds
# ============================================================
labels_test_pc, radii_test_pc, scores_test_pc, _ = cached_qsnet_infer_per_class(
    cache.get("test"), prototypes, q_by_class, p=noise_rate, L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF,
    zero_day=ZERO_DAY, device=DEVICE, batch_size=256)
labels_zday_pc, radii_zday_pc, scores_zday_pc, _ = cached_qsnet_infer_per_class(
    cache.get("zeroday"), prototypes, q_by_class, p=noise_rate, L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF,
    zero_day=ZERO_DAY, device=DEVICE, batch_size=256)

tp_pc, n_pc = int(np.sum(labels_zday_pc == ZERO_DAY)), len(labels_zday_pc)
fp_pc, m_pc = int(np.sum(labels_test_pc == ZERO_DAY)), len(labels_test_pc)
TPR_zeroday_pc, FPR_known_pc = tp_pc/n_pc, fp_pc/m_pc
tpr_pc_lo, tpr_pc_hi = clopper_pearson_ci(tp_pc, n_pc, alpha=0.05)
fpr_pc_lo, fpr_pc_hi = clopper_pearson_ci(fp_pc, m_pc, alpha=0.05)
print(f"[per-class] TPR={TPR_zeroday_pc:.4f} CI[{tpr_pc_lo:.4f},{tpr_pc_hi:.4f}]  "
      f"FPR={FPR_known_pc:.4f} CI[{fpr_pc_lo:.4f},{fpr_pc_hi:.4f}]")

# ============================================================
# SECTION 12b — Global vs Mondrian vs Hybrid comparison
# ============================================================
comparison_rows = [
    {"method": "Global", "TPR": TPR_zeroday, "TPR_lo": tpr_lo, "TPR_hi": tpr_hi,
     "FPR": FPR_known, "FPR_lo": fpr_lo, "FPR_hi": fpr_hi},
    {"method": "Mondrian", "TPR": TPR_zeroday_pc, "TPR_lo": tpr_pc_lo, "TPR_hi": tpr_pc_hi,
     "FPR": FPR_known_pc, "FPR_lo": fpr_pc_lo, "FPR_hi": fpr_pc_hi},
]
if USE_ISO_HYBRID:
    comparison_rows.append({"method": "Hybrid", "TPR": TPR_hybrid, "TPR_lo": tpr_h_lo, "TPR_hi": tpr_h_hi,
                             "FPR": FPR_hybrid, "FPR_lo": fpr_h_lo, "FPR_hi": fpr_h_hi})
df_comparison = pd.DataFrame(comparison_rows)
print(df_comparison.to_string(index=False))
savecsv(df_comparison, "part18_global_mondrian_hybrid_comparison")

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(df_comparison)); w = 0.35
tpr_err = [df_comparison["TPR"]-df_comparison["TPR_lo"], df_comparison["TPR_hi"]-df_comparison["TPR"]]
fpr_err = [df_comparison["FPR"]-df_comparison["FPR_lo"], df_comparison["FPR_hi"]-df_comparison["FPR"]]
ax.bar(x-w/2, df_comparison["TPR"], w, yerr=tpr_err, capsize=5, label="TPR", color="seagreen")
ax.bar(x+w/2, df_comparison["FPR"], w, yerr=fpr_err, capsize=5, label="FPR", color="firebrick")
ax.axhline(DEFAULT_ALPHA, color="black", linestyle="--", label=f"alpha={DEFAULT_ALPHA}")
ax.set_xticks(x); ax.set_xticklabels(df_comparison["method"]); ax.set_ylim(0,1.05); ax.legend()
plt.tight_layout(); savefig("part18_comparison")
safe_empty_cache()

[global] TPR=0.0087 CI[0.0071,0.0107]  FPR=0.0490 CI[0.0460,0.0522]  class_acc=0.7028
[per-class] TPR=0.0370 CI[0.0335,0.0407]  FPR=0.1152 CI[0.1107,0.1199]
  method      TPR   TPR_lo   TPR_hi      FPR   FPR_lo   FPR_hi
  Global 0.008740 0.007085 0.010663 0.049039 0.046002 0.052216
Mondrian 0.036963 0.033512 0.040662 0.115236 0.110715 0.119876
  Hybrid 0.371905 0.362856 0.381022 0.047079 0.044102 0.050197
  saved table -> /kaggle/working/quantum-sentinel/tables/part18_global_mondrian_hybrid_comparison.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/part18_comparison.png


In [16]:
# ============================================================
# DAY 25 — RQ3: disentangling adversarial-known from true zero-day
# ============================================================
eps_l2_sweep = np.linspace(0.0, max(abs(EPS_STAR_GLOBAL)*2.5, 1.0), 8)
eps_inf_sweep = [l2_to_linf_budget(e, d=num_qubits) for e in eps_l2_sweep]
rq3_idx = np.where(accepted_known)[0][:250]
A_known_rq3 = A_test[rq3_idx]
grad_sign_rq3 = run_batched_safely(fgsm_gradient_sign, A_known_rq3, theta_device, prototypes,
                                    forward_circuit, device=DEVICE, proto_stack=proto_stack_main,
                                    batch_size=8, min_batch=1, label="fgsm_grad[rq3]")

rq3_rows = []
for e_l2, e_inf in zip(eps_l2_sweep, eps_inf_sweep):
    if e_inf == 0.0:
        infer_out = cached_qsnet_infer(cache.slice("test", rq3_idx), prototypes, q_final, p=noise_rate,
                                        L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF, zero_day=ZERO_DAY,
                                        device=DEVICE, batch_size=256)
    elif grad_sign_rq3 is None:
        infer_out = None
    else:
        X_adv = apply_fgsm_perturbation(A_known_rq3, grad_sign_rq3, e_inf, device=DEVICE,
                                         x_min=0.0, x_max=float(np.pi))
        infer_out = run_batched_safely(qsnet_infer_batch, X_adv.cpu().numpy(), theta_device, prototypes,
                                        q_final, forward_circuit, p=noise_rate, L_phi=L_PHI_ANALYTIC,
                                        Cf=DEFAULT_CF, zero_day=ZERO_DAY, device=DEVICE,
                                        batch_size=8, min_batch=1, label=f"rq3_infer[eps={e_inf:.3f}]")
    reject_rate = float(np.mean(infer_out[0] == ZERO_DAY)) if infer_out is not None else np.nan
    rq3_rows.append({"eps_l2": e_l2, "adv_known_reject_rate": reject_rate})
    safe_empty_cache()

df_rq3 = pd.DataFrame(rq3_rows).dropna(subset=["adv_known_reject_rate"])
savecsv(df_rq3, "day25_rq3_disentanglement")

if not df_rq3.empty:
    eps_plot, is_safe = is_eps_safe_to_plot(EPS_STAR_GLOBAL)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df_rq3["eps_l2"], df_rq3["adv_known_reject_rate"], marker="o", label="adversarial-known reject rate")
    ax.axhline(TPR_zeroday, color="firebrick", linestyle="--", label=f"true zero-day reject rate ({TPR_zeroday:.2f})")
    if is_safe:
        ax.axvline(eps_plot, color="green", linestyle=":", label=f"predicted eps*={eps_plot:.3f}")
    ax.set_ylim(-0.02, 1.05); ax.legend(); ax.set_title("RQ3: disentangling adversarial-known from zero-day")
    plt.tight_layout(); savefig("day25_rq3_disentanglement")

  saved table -> /kaggle/working/quantum-sentinel/tables/day25_rq3_disentanglement.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/day25_rq3_disentanglement.png


In [17]:
# ============================================================
# Save the full Days 15-25 results bundle
# ============================================================
theory_validation = {
    "prop1_max_residual": max_resid, "lipschitz_analytic": L_PHI_ANALYTIC,
    "certified_radius": {"mean": float(certified_radii_correct.mean()),
                          "median": float(np.median(certified_radii_correct)),
                          "min": float(certified_radii_correct.min()),
                          "max": float(certified_radii_correct.max())},
    "F_in": F_IN, "F_out": F_OUT, "epsilon_star_strict": eps_star_strict,
    "epsilon_star_strict_is_vacuous": bool(eps_star_strict <= 0),
    "epsilon_beta": eps_beta, "doubly_robust_epsilon": eps_robust,
    "per_class_prop2": df_prop2_perclass.to_dict(orient="records"),
    "day19_acceptance_curve": df_day19.to_dict(orient="records"),
    "empirical_breakpoint_l2": empirical_breakpoint_l2, "H3_holds": bool(h3_holds),
    "day20_p_sweep": df_day20.to_dict(orient="records"),
    "day21_per_dataset": df_day21.to_dict(orient="records"),
    "F_in_global": float(F_IN_GLOBAL), "epsilon_star_global": EPS_STAR_GLOBAL,
    "RQ1_detection_global": {"TPR": TPR_zeroday, "TPR_ci95": [tpr_lo, tpr_hi],
                              "FPR": FPR_known, "FPR_ci95": [fpr_lo, fpr_hi]},
    "RQ1_detection_mondrian": {"TPR": TPR_zeroday_pc, "TPR_ci95": [tpr_pc_lo, tpr_pc_hi],
                                "FPR": FPR_known_pc, "FPR_ci95": [fpr_pc_lo, fpr_pc_hi]},
    "comparison_table": df_comparison.to_dict(orient="records"),
    "RQ3_disentanglement": df_rq3.to_dict(orient="records"),
    "cache_memory_report": cache.memory_report(),
}
savejson(theory_validation, "theory_validation_days15_25")

  saved json -> /kaggle/working/quantum-sentinel/logs/theory_validation_days15_25.json


PosixPath('/kaggle/working/quantum-sentinel/logs/theory_validation_days15_25.json')

In [18]:
# ============================================================
# DAY 26 (a) — Certified-accuracy-at-radius curve (RQ4)
# ============================================================
radius_grid = np.linspace(0, np.percentile(certified_radii_correct, 99), 40)
cert_acc = [float(np.mean((labels_test_final == y_test) & (radii_test_final >= r))) for r in radius_grid]
df_cert_acc = pd.DataFrame({"radius": radius_grid, "certified_accuracy": cert_acc})
savecsv(df_cert_acc, "day26_certified_accuracy_at_radius")

plt.figure(figsize=(7.5, 5))
plt.plot(radius_grid, cert_acc, marker=".", color="teal")
plt.xlabel("certified radius r"); plt.ylabel("certified accuracy (fraction with R>=r, correctly labeled)")
plt.title("RQ4: certified-accuracy-at-radius"); plt.tight_layout()
savefig("day26_certified_accuracy_curve")

# ============================================================
# DAY 26 (b) — Noise-rate p ablation (certified vs empirical), reusing Day 20
# ============================================================
df_noise_ablation = df_day20.copy()
df_noise_ablation["h3_holds_row"] = df_noise_ablation["empirical_breakpoint_l2"] >= df_noise_ablation["epsilon_star_correct"]
savecsv(df_noise_ablation, "day26_noise_rate_ablation")
print(df_noise_ablation.to_string(index=False))

  saved table -> /kaggle/working/quantum-sentinel/tables/day26_certified_accuracy_at_radius.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/day26_certified_accuracy_curve.png
  saved table -> /kaggle/working/quantum-sentinel/tables/day26_noise_rate_ablation.csv
   p     F_in    F_out  epsilon_star_correct  epsilon_star_naive_wrong  empirical_breakpoint_l2  h3_holds_row
0.00 0.319602 0.947719             -0.298424                 -0.281909                 0.122474          True
0.05 0.776721 0.991896             -0.218155                 -0.296746                      inf          True
0.10 0.916027 0.997708             -0.147713                 -0.313232                      inf          True
0.20 0.992091 0.999820             -0.052225                 -0.352386                      inf          True


In [19]:
# ============================================================
# DAY 26 (c) — MAQT ablation: CE-only baseline vs MAQT
# *** THIS BLOCK TRAINS A NEW MODEL (train_plain_vqc). ***
# Everything else in this notebook reuses your existing MAQT checkpoint;
# this is the one exception, because there is no CE-only checkpoint saved yet.
# ============================================================
RUN_MAQT_ABLATION = False   # <-- flip to True only when you want to spend the compute

if RUN_MAQT_ABLATION:
    PLAIN_CKPT_DIR = CKPT_DIR / "plain_vqc"
    theta_plain, head_plain, history_plain = train_plain_vqc(
        A_train, y_train, n_classes=num_classes, n_qubits=num_qubits, n_layers=num_layers,
        forward_circuit=forward_circuit, device=DEVICE,
        epochs=10, lr=0.05, batch_size=8, use_weighted_sampler=True,
        checkpoint_dir=str(PLAIN_CKPT_DIR), log_dir=str(LOG_DIR), notebook_name="plain_vqc",
        seed=SEED, verbose=True,
    )
    torch.save({"theta": theta_plain.detach().cpu(),
                "head_state_dict": head_plain.state_dict(),
                "history": history_plain}, CKPT_DIR / "plain_vqc_final.pt")

    epsilons = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2]
    ablation_rows = robustness_ablation(
        A_test, y_test, theta_device, classifier_head, theta_plain.to(DEVICE), head_plain,
        forward_circuit, epsilons, DEVICE, attack_fn=fgsm_attack, batch_size=256,
        x_min=0.0, x_max=float(np.pi),
    )
    df_maqt_ablation = pd.DataFrame(ablation_rows)
    savecsv(df_maqt_ablation, "day26_maqt_vs_ce_ablation")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df_maqt_ablation["eps"], df_maqt_ablation["maqt_acc"], marker="o", label="MAQT")
    ax.plot(df_maqt_ablation["eps"], df_maqt_ablation["plain_acc"], marker="s", label="CE-only")
    ax.set_xlabel("FGSM epsilon"); ax.set_ylabel("accuracy"); ax.legend()
    ax.set_title("MAQT ablation: geometry-shaping benefit under attack")
    plt.tight_layout(); savefig("day26_maqt_vs_ce_ablation")
else:
    print("MAQT ablation skipped (RUN_MAQT_ABLATION=False) — set True to train the CE-only baseline.")

MAQT ablation skipped (RUN_MAQT_ABLATION=False) — set True to train the CE-only baseline.


In [20]:
# ============================================================
# DAY 26 (d) — Freeze all artefacts + one-command re-run confirmation
# ============================================================
frozen_artefacts = {
    "theta": theta_star, "head_state_dict": head_state_dict,
    "prototypes": {c: rho.detach().cpu() for c, rho in prototypes.items()},
    "q_threshold": q_final, "q_by_class": q_by_class, "L_phi_analytic": L_PHI_ANALYTIC,
    "noise_p": noise_rate, "num_qubits": num_qubits, "num_layers": num_layers,
    "class_names": class_names, "checkpoint_source": str(CHECKPOINT_PATH),
    "cache_source": str(CACHE_LATEST), "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
FROZEN_PATH = CKPT_DIR / "qsnet_frozen_day26.pt"
torch.save(frozen_artefacts, FROZEN_PATH)
print(f"Frozen artefacts -> {FROZEN_PATH}")

# one-command re-run check: reload from disk and re-verify a single known/zero-day sample
# _frozen = torch.load(FROZEN_PATH, map_location="cpu", weights_only=False)
# _protos_reload = {c: v.to(DEVICE) for c, v in _frozen["prototypes"].items()}
# rho_chk = cache.get("test")["rho"][0].to(DEVICE)
# with torch.no_grad():
#     f_chk = {c: float(fidelity(rho_chk, _protos_reload[c])) for c in _protos_reload}
# print("Re-run check (fidelities from reloaded frozen artefacts):", f_chk)

Frozen artefacts -> /kaggle/working/quantum-sentinel/checkpoints/qsnet_frozen_day26.pt


In [21]:
# ============================================================
# DAY 27 (a) — Figure 1: trust-region visualisation
# (2D projection of Hilbert-space prototypes + samples)
# ============================================================
N_VIZ2 = 600
rng2 = np.random.default_rng(SEED)
idx_t = rng2.choice(len(A_test), size=min(N_VIZ2, len(A_test)), replace=False)
idx_z = rng2.choice(len(A_zeroday), size=min(N_VIZ2, len(A_zeroday)), replace=False)

rho_known = cache.get("test")["rho"][idx_t]
rho_zday  = cache.get("zeroday")["rho"][idx_z]
proto_stack_t = torch.stack([prototypes[c].cpu() for c in sorted(prototypes)], dim=0)
rho_all_fig1 = torch.cat([rho_known, rho_zday, proto_stack_t], dim=0)

_, feats_fig1 = fidelity_to_prototypes_matrix(rho_all_fig1, prototypes)
emb_fig1 = pca_2d(feats_fig1)
n_known, n_zday_, n_proto = len(idx_t), len(idx_z), proto_stack_t.shape[0]
emb_known, emb_zday, emb_proto = emb_fig1[:n_known], emb_fig1[n_known:n_known+n_zday_], emb_fig1[n_known+n_zday_:]

labels_known_fig1 = y_test[idx_t]
plt.figure(figsize=(9, 8))
palette = plt.cm.tab10(np.linspace(0, 1, num_classes))
for c in range(num_classes):
    m = labels_known_fig1 == c
    plt.scatter(emb_known[m, 0], emb_known[m, 1], s=12, alpha=0.5, color=palette[c], label=class_names[c])
plt.scatter(emb_zday[:, 0], emb_zday[:, 1], s=16, alpha=0.6, color="black", marker="x", label="zero-day")
for i, c in enumerate(sorted(prototypes)):
    plt.scatter(*emb_proto[i], s=260, marker="*", color=palette[c], edgecolor="black", linewidth=1.2, zorder=5)
plt.legend(fontsize=8, loc="best")
plt.title("Figure 1 — Trust-region visualisation\n(2D PCA of Hilbert-space fidelity vectors; stars = class prototypes)")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.tight_layout()
savefig("figure1_trust_regions")

# ============================================================
# DAY 27 (b) — Figure 3: disentanglement (separation AUROC vs attack budget)
# ============================================================
auroc_rows = []
for e_l2, e_inf in zip(eps_l2_sweep, eps_inf_sweep):
    if e_inf == 0.0:
        rho_known_adv = cache.slice("test", rq3_idx)["rho"]
    elif grad_sign_rq3 is None:
        continue
    else:
        X_adv = apply_fgsm_perturbation(A_known_rq3, grad_sign_rq3, e_inf, device=DEVICE,
                                         x_min=0.0, x_max=float(np.pi))
        with torch.no_grad():
            _, rho_known_adv = forward_circuit(X_adv, theta_device)
        rho_known_adv = rho_known_adv.detach().cpu()

    s_known = 1.0 - np.array([
        max(float(fidelity(rho_known_adv[i].to(DEVICE), prototypes[c])) for c in prototypes)
        for i in range(rho_known_adv.shape[0])
    ])
    s_zday = scores_zday_final  # from Day 23-24, fixed zero-day nonconformity scores

    y_bin = np.concatenate([np.zeros(len(s_known)), np.ones(len(s_zday))])
    s_all = np.concatenate([s_known, s_zday])
    sep_auroc = roc_auc_score(y_bin, s_all)
    auroc_rows.append({"eps_l2": e_l2, "separation_auroc": sep_auroc})

df_fig3 = pd.DataFrame(auroc_rows)
savecsv(df_fig3, "figure3_disentanglement_auroc")

plt.figure(figsize=(8, 5))
plt.plot(df_fig3["eps_l2"], df_fig3["separation_auroc"], marker="o", color="darkorange")
plt.axhline(0.5, color="gray", linestyle=":", label="chance (0.5)")
eps_plot, is_safe = is_eps_safe_to_plot(EPS_STAR_GLOBAL)
if is_safe:
    plt.axvline(eps_plot, color="green", linestyle="--", label=f"Proposition-2 threshold eps*={eps_plot:.3f}")
plt.ylim(0.4, 1.02); plt.xlabel("attack budget epsilon (L2)"); plt.ylabel("separation AUROC (known-adv vs zero-day)")
plt.title("Figure 3 — Disentanglement: separation AUROC vs attack budget")
plt.legend(); plt.tight_layout()
savefig("figure3_disentanglement")

  saved figure -> /kaggle/working/quantum-sentinel/figures/figure1_trust_regions.png
  saved table -> /kaggle/working/quantum-sentinel/tables/figure3_disentanglement_auroc.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/figure3_disentanglement.png


In [22]:
# ============================================================
# DAY 28 (a) — Table C: certified + empirical accuracy under FGSM/PGD, across eps and noise p
# ============================================================
epsilons_tableC = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2]
p_values_tableC = [0.0, noise_rate, 0.1, 0.2]

table_c_rows = []
for p_val in p_values_tableC:
    fc_p = build_forward_circuit(dev, num_qubits, num_layers, noise_rate=p_val, reupload=reupload)
    for eps in epsilons_tableC:
        # empirical accuracy under FGSM (classifier-head accuracy on perturbed known samples)
        res_fgsm = eval_attacked(fgsm_attack, A_test, y_test, theta_device, classifier_head, fc_p,
                                  DEVICE, batch_size=32, eps=eps, x_min=torch.tensor(0.0, device=DEVICE), x_max=torch.tensor(np.pi, device=DEVICE))
        # certified accuracy at this eps (radius >= eps, correct label), from the p=noise_rate certificate
        cert_acc_eps = float(np.mean((labels_test_final == y_test) & (radii_test_final >= eps)))
        table_c_rows.append({"p": p_val, "eps": eps,
                              "empirical_acc_fgsm": res_fgsm["acc"],
                              "empirical_macro_f1_fgsm": res_fgsm["macro_f1"],
                              "certified_acc": cert_acc_eps})
    safe_empty_cache()

df_table_c = pd.DataFrame(table_c_rows)
savecsv(df_table_c, "day28_table_c_robustness")
print(df_table_c.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5.5))
for p_val, grp in df_table_c.groupby("p"):
    ax.plot(grp["eps"], grp["empirical_acc_fgsm"], marker="o", label=f"empirical (p={p_val:.2f})")
ax.plot(df_table_c.query("p == @noise_rate")["eps"],
        df_table_c.query("p == @noise_rate")["certified_acc"],
        marker="s", linestyle="--", color="black", label="certified (p=trained noise)")
ax.set_xlabel("attack budget epsilon"); ax.set_ylabel("accuracy")
ax.set_title("Table C — certified vs empirical accuracy across eps and noise p")
ax.legend(fontsize=8); plt.tight_layout()
savefig("day28_table_c_robustness")

# ============================================================
# DAY 28 (b) — Certified-radius distribution plots (all datasets that have labels)
# ============================================================
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(certified_radii_correct, bins=40, alpha=0.8, color="teal", label="test (correct & accepted)")
ax.axvline(certified_radii_correct.mean(), color="red", linestyle="--",
           label=f"mean={certified_radii_correct.mean():.3f}")
ax.axvline(np.median(certified_radii_correct), color="orange", linestyle=":",
           label=f"median={np.median(certified_radii_correct):.3f}")
ax.set_xlabel("certified radius R"); ax.set_title("Day 28 — final certified-radius distribution")
ax.legend(); plt.tight_layout()
savefig("day28_certified_radius_final")

  saved table -> /kaggle/working/quantum-sentinel/tables/day28_table_c_robustness.csv
   p  eps  empirical_acc_fgsm  empirical_macro_f1_fgsm  certified_acc
0.00 0.00            0.692528                 0.390839       0.668379
0.00 0.02            0.634380                 0.352638       0.177408
0.00 0.05            0.453265                 0.246548       0.002065
0.00 0.10            0.136048                 0.118098       0.000106
0.00 0.15            0.047026                 0.067667       0.000000
0.00 0.20            0.027326                 0.040233       0.000000
0.01 0.00            0.709527                 0.406969       0.668379
0.01 0.02            0.628131                 0.351492       0.177408
0.01 0.05            0.415559                 0.235557       0.002065
0.01 0.10            0.137002                 0.122228       0.000106
0.01 0.15            0.043690                 0.065122       0.000000
0.01 0.20            0.030133                 0.041484       0.000000
0.10

In [23]:
# ============================================================
# Save the Day 26-28 results bundle
# ============================================================
final_bundle = {
    "day26_certified_accuracy_curve": df_cert_acc.to_dict(orient="records"),
    "day26_noise_rate_ablation": df_noise_ablation.to_dict(orient="records"),
    "day26_maqt_ablation_ran": bool(RUN_MAQT_ABLATION),
    "day27_figure3_auroc": df_fig3.to_dict(orient="records"),
    "day28_table_c": df_table_c.to_dict(orient="records"),
    "frozen_artefacts_path": str(FROZEN_PATH),
}
savejson(final_bundle, "results_days26_28")
print("\nAll outputs written under:", OUT_ROOT)
print(" figures  ->", FIG_DIR)
print(" tables   ->", TABLE_DIR)
print(" logs     ->", LOG_DIR)
print(" ckpts    ->", CKPT_DIR)

  saved json -> /kaggle/working/quantum-sentinel/logs/results_days26_28.json

All outputs written under: /kaggle/working/quantum-sentinel
 figures  -> /kaggle/working/quantum-sentinel/figures
 tables   -> /kaggle/working/quantum-sentinel/tables
 logs     -> /kaggle/working/quantum-sentinel/logs
 ckpts    -> /kaggle/working/quantum-sentinel/checkpoints
